# Divvy Data Cleaning
- Loaded and combined the files 32files and 16,304,359 rows

- Removed duplicates 35 rows

- Removed rides with no end location 16,951

- Removed outliers and saved them in a new dataframe

- Remaining rows 15,851,351 (97.2% of original dataset)

**Reasoning** 

- Duplicates corrupt the data

- Kept rides with missing station names because they still count for the ride

- Removed rides with no end location because those are rides where the user probably forgot to return the bike to the station or end their rides properly.

- Rides under 1 minute are dropped because can't have a negative ride

- Rides over 180 minutes are dropped, casual rides rises with ride length over 180 are mostly docking error rather than real trips

**Column and Meaning**

ride_id - unique id of each ride

rideable_type - classic bike or electric

started_at/ended_at - start and end date and time

start_station_name and end_station_name - sometimes null may be blank for e-bikes

start_lat and end_lat - cannot be null, all rides most start and end loaction

member_casual - member type, member or casual

ride_length - added column to have the ride duration in minute

**Variables and what they hold**

files - the list of the 32 csv file paths

df - the dataframe

is_outlier - True/False with rides under a minute and over 180 minutes

- outliers - the removed outlier rides, kept for reference



In [25]:
import pandas as pd 
import numpy as np 
import glob

files = glob.glob("data/*.csv") # merge all files into a single folder
print(len(files))

32


In [26]:
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True) # concat all 32 files into one
print(len(df))

16304359


In [27]:
df.head() # check the earliest data loaded

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,C1D650626C8C899A,electric_bike,2024-01-12 15:30:27,2024-01-12 15:37:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.903267,-87.634737,41.889177,-87.638506,member
1,EECD38BDB25BFCB0,electric_bike,2024-01-08 15:45:46,2024-01-08 15:52:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902937,-87.634440,41.889177,-87.638506,member
2,F4A9CE78061F17F7,electric_bike,2024-01-27 12:27:19,2024-01-27 12:35:19,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902951,-87.634470,41.889177,-87.638506,member
3,0A0D9E15EE50B171,classic_bike,2024-01-29 16:26:17,2024-01-29 16:56:06,Wells St & Randolph St,TA1305000030,Larrabee St & Webster Ave,13193,41.884295,-87.633963,41.921822,-87.644140,member
4,33FFC9805E3EFF9A,classic_bike,2024-01-31 05:43:23,2024-01-31 06:09:35,Lincoln Ave & Waveland Ave,13253,Kingsbury St & Kinzie St,KA1503000043,41.948797,-87.675278,41.889177,-87.638506,member


In [28]:
df.tail() # check that the most recent data loaded

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
16304354,7883C5E2B66C0C44,electric_bike,2026-09-18 17:01:46.488,2026-09-18 17:08:01.040,NaN,NaN,NaN,NaN,41.890000,-87.650000,41.88,-87.64,member
16304355,852196D2C54E52DB,electric_bike,2026-09-16 17:41:53.199,2026-09-16 17:49:20.229,NaN,NaN,NaN,NaN,41.950000,-87.650000,41.93,-87.65,member
16304356,FC1560800DCC3F82,electric_bike,2026-09-29 18:28:35.801,2026-09-29 18:36:49.157,Hudson Ave & Armitage Ave,CHI02110,NaN,NaN,41.918180,-87.640000,41.94,-87.64,member
16304357,00E83F17163FB3DD,electric_bike,2026-09-18 12:43:56.439,2026-09-18 12:48:28.484,Blue Island Ave & 18th St,CHI00443,NaN,NaN,41.857556,-87.661535,41.85,-87.67,member
16304358,4D562F64F62AABCC,electric_bike,2026-09-19 14:49:56.560,2026-09-19 14:58:57.981,Clarendon Ave & Junior Ter,CHI00395,NaN,NaN,41.961004,-87.649603,41.95,-87.65,member


In [29]:
first_cols = list(pd.read_csv(files[0], nrows=0).columns)

for f in files:
    cols = list(pd.read_csv(f, nrows=0).columns)
    if cols != first_cols:
        print("DIFFERENT:", f)
        print(cols)

print("Check finished")  # ensuring all column names are the same through the years

Check finished


In [30]:
is_dupe = df.duplicated(keep=False) # find duplicated row, marking every copy so we can see the original and twin side by side
dupes = df[is_dupe] # keep the only rows that have duplicates
dupes.sort_values("ride_id").head(40) # align the matching copies with the next so easy to compare

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
12176162,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,NaN,NaN,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member
12917016,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,NaN,NaN,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member
12297494,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,NaN,NaN,41.886871,-87.648089,NaN,NaN,member
12640075,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,NaN,NaN,41.886871,-87.648089,NaN,NaN,member
12532083,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual
12116721,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual
12048911,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member
12798325,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member
12307834,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,NaN,NaN,41.954690,-87.673930,41.970000,-87.670000,casual
12996765,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,NaN,NaN,41.954690,-87.673930,41.970000,-87.670000,casual


In [31]:
rows_before = len(df) # number of row before dropping duplicate
df = df.drop_duplicates() # number of rows after dropping duplicate
print("Duplicate removed", rows_before - len(df)) # print how many rows were dropped

Duplicate removed 35


In [32]:
print(len(df))
df.isna().sum() # see the total number of missssig cells per column

16304324


ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    3275366
start_station_id      3275366
end_station_name      3410429
end_station_id        3410429
start_lat                   0
start_lng                   0
end_lat                 16951
end_lng                 16951
member_casual               0
dtype: int64

In [33]:
#drop rows with no start or end lat because every bike not be picked up or returned to a station. 

row_before_drop = len(df)
df = df.dropna(subset=['end_lat', 'end_lng'])
print('rows removed', row_before_drop - len(df))

rows removed 16951


In [34]:
print(len(df))

16287373


In [35]:
#craete  a new column called ride_length and see the mean and check for rows that greatly exceed the ride length

df['started_at'] = pd.to_datetime(df["started_at"], format='mixed') # change to datetime from string for calculations
df['ended_at'] = pd.to_datetime(df['ended_at'], format='mixed')

df['ride_length'] = (df['ended_at']- df['started_at']).dt.total_seconds() / 60 # time in minutes

In [41]:
# see the mean ride time
pd.set_option("display.float_format", "{:,.2f}".format) # so its set to regular notation
df['ride_length'].describe().apply(lambda x: f"{x :,.2f}") #show decimals instead of scientific notation and add commas to make it more readable


count    16,287,373.00
mean             14.79
std              29.93
min          -2,748.32
25%               5.46
50%               9.53
75%              16.75
max           1,509.37
Name: ride_length, dtype: object

In [42]:
print("Negative or zero:", (df["ride_length"] <= 0).sum())
print("Under 1 min:", (df["ride_length"] < 1).sum())
print("Over 60 min:", (df["ride_length"] > 60).sum())
print("Over 120 min:", (df["ride_length"] > 120).sum())

Negative or zero: 752
Under 1 min: 403780
Over 60 min: 351156
Over 120 min: 80636


In [50]:
df["member_casual"].value_counts(normalize=True)

member_casual
member   0.63
casual   0.37
Name: proportion, dtype: float64

In [ ]:
# compare lenght of rides between casual and member to know when to cut of outliers
valid_long = df[df["ride_length"] > 1]
bins = [0, 15, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 1600]
long_bins = pd.cut(valid_long["ride_length"], bins=bins)

pd.crosstab(long_bins, valid_long["member_casual"])

member_casual,casual,member
ride_length,,
"(0, 15]",3468918,7643922
"(15, 30]",1372654,1921504
"(30, 60]",620116,504940
"(60, 90]",161739,37855
"(90, 120]",60629,10297
"(120, 150]",26598,4745
"(150, 180]",14170,2881
"(180, 210]",5747,2153
"(210, 240]",2721,1486


In [55]:
before_outliers = len(df)

is_outlier = (df["ride_length"] < 1) | (df['ride_length'] > 180) # outliers
outliers = df[is_outlier] # save the bad rides
df = df[~is_outlier] #keep everything else

print(f"Before: {before_outliers:,}")
print(f"Outliers:  {len(outliers):,}")
print(f"Clean: {len(df):,}")

if len(outliers) + len(df) == before_outliers:
    print("All rows accounted for")
    
else:
    print("Error")

Before: 16,287,373
Outliers:  436,022
Clean: 15,851,351
All rows accounted for


In [56]:
# to export the data

df.to_parquet("data/divvy_clean_2024_2026.parquet", index=False)

In [57]:
df.to_csv("data/divvy_clean_2024_2026.csv", index=False)